---
title: "3.2.b Amenaza por sequía y estiaje"
short_title: "b. Amenaza"
subtitle: "Aguas bajas extremas del Amazonas, sequía meteorológica y relación con El Niño"
description: "Análisis de las vaciantes extremas del Amazonas en Leticia (IDEAM 1987–2026), índices SPI/SPEI de ERA5 y el índice ONI."
keywords: [sequía, estiaje, vaciante, Amazonas, SPI, SPEI, El Niño, Leticia]
---

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mromerot/leticia/blob/main/book/03_analisis/32_sequia/b_amenaza.ipynb)

In [ ]:
# Preparación del entorno. En Colab instala el paquete del proyecto desde GitHub (1–2 minutos);
# en una copia local usa el paquete ya instalado con `pip install -e .`.
import sys, subprocess
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "leticia @ git+https://github.com/mromerot/leticia"], check=True)
from leticia.libro import preparar, intentar
preparar()

En Leticia la sequía no se mide solo en lluvia: se mide en **metros de río**. Entre agosto y noviembre el
Amazonas baja a su mínimo y, cuando la vaciante es extrema, los barcos de carga no llegan al puerto, se
encarecen los alimentos y el combustible, se secan los lagos y la quebrada de la que se abastece el acueducto,
y mueren peces. Las vaciantes de 2023 y 2024 fueron las más severas registradas en la cuenca
[@espinoza2024; @costa2023spa], y el IDEAM reportó en septiembre de 2024 niveles por debajo de los mínimos
históricos en la estación de Leticia [@ideam2024comunicado100].

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from leticia import clima, fuentes, hidro, viz
from leticia.config import parametros

ph = parametros()["hidrologia"]
nivel = hidro.limpiar_nivel(fuentes.nivel_rio())
minimos = hidro.extremos_anuales(nivel, "min")

## 1. La vaciante de 2024 frente a la historia

In [ ]:
clim = hidro.climatologia_diaria(nivel[nivel.index.year < 2023], base=(1991, 2020))
fig, ax = plt.subplots(figsize=(10.5, 4))
x = clim.index
ax.fill_between(x, clim["min"] / 100, clim["max"] / 100, color=viz.GRIS, alpha=0.15, label="mín–máx 1988–2022")
ax.fill_between(x, clim["p10"] / 100, clim["p90"] / 100, color=viz.GRIS, alpha=0.3, label="p10–p90")
ax.plot(x, clim["p50"] / 100, color=viz.GRIS, lw=1.2, label="mediana")
for anio, color in ((2010, "#fdae61"), (2023, "#f46d43"), (2024, "#a50026"), (2025, "#4575b4"), (2026, "#313695")):
    s = nivel[nivel.index.year == anio]
    ax.plot(s.index.dayofyear, s / 100, color=color, lw=1.4, label=str(anio))
for u, et in ((ph["umbral_estiaje_cm"], "aguas muy bajas"), (ph["umbral_estiaje_critico_cm"], "aislamiento del puerto")):
    ax.axhline(u / 100, color="k", ls=":", lw=0.8)
    ax.text(5, u / 100 + 0.15, f"{u/100:.0f} m: {et} (ilustrativo)", fontsize=7)
ax.set_xticks([1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335])
ax.set_xticklabels(list("EFMAMJJASOND"))
ax.set_ylabel("nivel (m)")
ax.set_title("Nivel del Amazonas en Leticia: años recientes frente a 1988–2022")
ax.legend(ncol=4, fontsize=7.5, loc="lower center")
fig.tight_layout()

In [ ]:
viz.pie(viz.nota_fuente("nivel_rio_leticia"))

In [ ]:
m24 = nivel["2024"].min()
previo = nivel[nivel.index.year < 2023].min()
display(Markdown(f"Mínimo de 2024: **{m24/100:.2f} m** el {nivel['2024'].idxmin():%d-%m-%Y}. Mínimo de 1988–2022: "
                 f"{previo/100:.2f} m ({nivel[nivel.index.year < 2023].idxmin():%Y}). La vaciante de 2024 quedó "
                 f"**{(previo - m24)/100:.1f} m por debajo** del mínimo previo del registro."))

## 2. Mínimos anuales, tendencia y periodos de retorno

In [ ]:
mk = hidro.mann_kendall(minimos["valor"])
fig, ax = plt.subplots(figsize=(10, 3.6))
col = np.where(minimos["valor"] <= ph["umbral_estiaje_cm"], viz.ROJO, viz.TIERRA)
ax.bar(minimos.index, minimos["valor"] / 100, color=col)
t = np.array(minimos.index, dtype=float)
ax.plot(t, (np.median(minimos["valor"]) + mk["sen"] * (t - np.median(t))) / 100, color="k", lw=1, label="pendiente de Sen")
ax.axhline(ph["umbral_estiaje_cm"] / 100, color=viz.GRIS, ls="--", lw=1, label="6 m (ilustrativo)")
ax.set_ylabel("mínimo anual (m)")
ax.set_title("Nivel mínimo de aguas bajas por año (agosto–noviembre)")
ax.legend()
fig.tight_layout()
display(Markdown(f"Años con dato: **{mk['n']}**. Mann-Kendall: z = {mk['z']:.2f}, p = {mk['p']:.2f}; pendiente de Sen "
                 f"**{mk['sen']:.1f} cm/año** ({mk['sen']*10/100:.2f} m por década). Los cinco mínimos más bajos: "
                 + ", ".join(f"{a} ({v/100:.2f} m)" for a, v in minimos['valor'].nsmallest(5).items()) + "."))

In [ ]:
T = parametros()["periodos_retorno"]
sin24 = minimos.loc[minimos.index < 2023, "valor"].values
tab = pd.concat({"con 2023–2026": hidro.niveles_retorno_ic(minimos["valor"].values, T, minimos=True)["nivel"] / 100,
                 "solo hasta 2022": hidro.niveles_retorno_ic(sin24, T, minimos=True)["nivel"] / 100}, axis=1)
display(Markdown("**Nivel mínimo (m) que se alcanza en promedio una vez cada T años** (GEV ajustada a −mínimos):"))
display(tab.round(2))
p_sin = hidro.ajustar_gev(sin24, minimos=True)
from scipy import stats
p24 = stats.genextreme.sf(-m24, *p_sin)
cota = -(p_sin[1] + p_sin[2] / p_sin[0]) if p_sin[0] > 0 else -np.inf
texto = (f"retorno de **{1/p24:,.0f} años**".replace(",", ".") if p24 > 0 else
         f"probabilidad **nula**: la distribución ajustada con los datos hasta 2022 tiene un mínimo posible de ≈ {cota/100:.2f} m")
display(Markdown(f"Con la estadística disponible hasta 2022, una vaciante como la de 2024 ({m24/100:.2f} m) tenía {texto}. "
                 "Al incluir 2023–2026 la curva se desplaza y lo que parecía imposible pasa a tener un periodo de retorno de pocas "
                 "décadas. Es la firma de un **régimen que cambia**, no de un evento aislado."))

## 3. Días de aguas muy bajas por año

In [ ]:
dias6 = hidro.dias_bajo_umbral(nivel, ph["umbral_estiaje_cm"])
dias4 = hidro.dias_bajo_umbral(nivel, ph["umbral_estiaje_critico_cm"])
fig, ax = plt.subplots(figsize=(10, 3.3))
ax.bar(dias6.index, dias6, color=viz.TIERRA, label="< 6 m")
ax.bar(dias4.index, dias4, color=viz.ROJO, label="< 4 m")
ax.set_ylabel("días por año")
ax.set_title("Días con el río por debajo de los umbrales de navegación (ilustrativos)")
ax.legend()
fig.tight_layout()
display(pd.DataFrame({"días < 6 m": dias6, "días < 4 m": dias4}).tail(8).T)

Los umbrales de 6 m y 4 m en la regla de Leticia son **ilustrativos**: la Capitanía de Puerto y las empresas de
transporte usan calados y niveles propios. El indicador relevante es la tendencia: en 2023 y 2024 el río pasó
muchas más semanas en aguas muy bajas que en cualquier año anterior del registro.

## 4. Sequía meteorológica en Leticia (ERA5)

In [ ]:
era = fuentes.clima_diario().loc["1979":]   # era satelital: ERA5 es más confiable desde 1979
precip = era["precipitation_sum"]
spi3 = clima.spi(precip, 3, base=(1991, 2020))
spei3 = clima.balance_hidrico_spei(precip, era["et0_fao_evapotranspiration"], 3, base=(1991, 2020))
fig, ax = plt.subplots(figsize=(11, 3.4))
s = spei3["1980":]
ax.fill_between(s.index, 0, s, where=s < 0, color=viz.ROJO, alpha=0.7, label="déficit")
ax.fill_between(s.index, 0, s, where=s >= 0, color=viz.AZUL_RIO, alpha=0.6, label="exceso")
ax.axhline(-1.5, color="k", ls=":", lw=0.8)
ax.set_ylabel("SPEI-3")
ax.set_title("Balance hídrico estandarizado de 3 meses (lluvia − evapotranspiración) en Leticia")
ax.legend(loc="lower left")
fig.tight_layout()
peores = spei3.nsmallest(6)
display(Markdown("Meses con el SPEI-3 más bajo: " + ", ".join(f"{i:%m/%Y} ({v:.1f})" for i, v in peores.items()) + "."))

In [ ]:
viz.pie(viz.nota_fuente("clima_diario") + " SPI según McKee et al. (1993); SPEI según Vicente-Serrano et al. (2010).")

El SPEI resta a la lluvia la evapotranspiración de referencia, de modo que también captura el efecto del calor:
en 2023–2024 no solo llovió menos, sino que hizo mucho más calor y el agua se evaporó más rápido. Según el
análisis de atribución de World Weather Attribution, el cambio climático hizo unas 30 veces más probable la
sequía agrícola de 2023 en la cuenca, sobre todo por el aumento de temperatura [@clarke2024wwa]. La lluvia sobre
Leticia explica solo una parte de la vaciante: el nivel del río depende de la lluvia en toda la cuenca alta.

## 5. El Niño y las vaciantes

In [ ]:
oni = fuentes.oni()
oni_jas = oni[oni.index.month == 8]           # trimestre julio–agosto–septiembre
oni_jas.index = oni_jas.index.year
df = pd.concat({"minimo_m": minimos["valor"] / 100, "oni_jas": oni_jas}, axis=1).dropna()
r = df.corr().iloc[0, 1]
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.scatter(df["oni_jas"], df["minimo_m"], c=df.index, cmap="viridis", s=30)
for a in (1998, 2010, 2015, 2023, 2024):
    if a in df.index:
        ax.annotate(str(a), (df.loc[a, "oni_jas"], df.loc[a, "minimo_m"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("ONI (julio–septiembre)")
ax.set_ylabel("mínimo anual en Leticia (m)")
ax.set_title(f"El Niño y el mínimo del río (r = {r:.2f})")
fig.tight_layout()
display(Markdown(f"ONI más reciente: **{oni.iloc[-1]:+.2f} °C** ({oni.index[-1]:%m/%Y}, trimestre centrado)."))

In [ ]:
viz.pie(viz.nota_fuente("indices_oceanicos"))

**Lectura.** Los años de El Niño tienden a tener vaciantes más bajas, pero la relación es imperfecta: la sequía de
2005 y la de 2010 respondieron sobre todo al calentamiento del Atlántico Norte tropical, y la de 2023–2024
combinó El Niño, un Atlántico muy cálido y el calentamiento global [@marengo2021spa22; @espinoza2024]. Un
El Niño fuerte en curso es, por tanto, una **señal de alerta** para la vaciante siguiente, no una certeza.